# 1. Построение каталога драйверов

Ноутбук последовательно анализирует Markdown-кейсы из `data/train`, сопоставляет их с текущим каталогом, применяет валидные дополнения и сохраняет подробный JSONL-аудит. Во время обработки выводятся номер текущего кейса, общее количество, прошедшее время, ETA и счетчики успешных/пропущенных кейсов. Ошибка отдельного кейса печатается с traceback, после чего обработка продолжается со следующего кейса. Перед запуском создайте `.env` из `.env.example`.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
%cd {ROOT}


/home/elwis/Projects/evaluation-drivers


/home/elwis/Projects/venv/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
from src.config import Settings
from src.file_io import discover_markdown_cases, load_catalog
from src.catalog_builder import run_catalog_build

settings = Settings.from_env()
cases = discover_markdown_cases(settings.train_cases_dir)
print(f"Train cases: {len(cases)}")
print(f"Catalog: {settings.driver_catalog_path}")
print(f"Model: {settings.openai_model}")


Train cases: 1
Catalog: artifacts/drivers/evaluation_drivers.json
Model: gpt-4.1-mini


In [3]:
catalog = run_catalog_build(settings)
print(f"Catalog v{catalog.catalog_version}: {len(catalog.drivers)} drivers")


Catalog build: 61 cases to process
[1/61] Processing case_000...
[1/61] Done | elapsed: 24s | ETA: 1463s | successful: 1 | failed: 0
[2/61] Processing case_001...
[2/61] ERROR in case_001: ValidationError: 1 validation error for DriverCatalog
drivers.3
  Value error, numeric ranges overlap in data.annotation_volume [type=value_error, input_value={'driver_id': 'data.annot...ntroduced_by_cases': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
[2/61] Done | elapsed: 49s | ETA: 1442s | successful: 1 | failed: 1
[3/61] Processing case_002...


Traceback (most recent call last):
  File "/home/elwis/Projects/evaluation-drivers/src/catalog_builder.py", line 99, in run_catalog_build
    updated_catalog, decisions = apply_catalog_analysis(catalog, analysis, case_id)
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/elwis/Projects/evaluation-drivers/src/catalog_builder.py", line 71, in apply_catalog_analysis
    return DriverCatalog.model_validate(updated.model_dump()), decisions
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/elwis/Projects/venv/lib/python3.12/site-packages/pydantic/main.py", line 732, in model_validate
    return cls.__pydantic_validator__.validate_python(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
pydantic_core._pydantic_core.ValidationError: 1 validation error for DriverCatalog
drivers.3
  Value error, numeric ranges overlap in data.annotation_volume [type=value_error, input_value={'driver_id': 'data.annot...ntroduced_by_ca

[3/61] Done | elapsed: 67s | ETA: 1287s | successful: 2 | failed: 1
[4/61] Processing case_003...
[4/61] Done | elapsed: 90s | ETA: 1283s | successful: 3 | failed: 1
[5/61] Processing case_004...
[5/61] Done | elapsed: 105s | ETA: 1177s | successful: 4 | failed: 1
[6/61] Processing case_006...
[6/61] Done | elapsed: 126s | ETA: 1156s | successful: 5 | failed: 1
[7/61] Processing case_007...
[7/61] Done | elapsed: 142s | ETA: 1093s | successful: 6 | failed: 1
[8/61] Processing case_008...
[8/61] Done | elapsed: 176s | ETA: 1166s | successful: 7 | failed: 1
[9/61] Processing case_009...
[9/61] Done | elapsed: 228s | ETA: 1316s | successful: 8 | failed: 1
[10/61] Processing case_010...
[10/61] Done | elapsed: 244s | ETA: 1245s | successful: 9 | failed: 1
[11/61] Processing case_011...
[11/61] Done | elapsed: 265s | ETA: 1203s | successful: 10 | failed: 1
[12/61] Processing case_012...
[12/61] Done | elapsed: 285s | ETA: 1163s | successful: 11 | failed: 1
[13/61] Processing case_013...
[13

In [4]:
import pandas as pd
pd.DataFrame([
    {"driver_id": d.driver_id, "name": d.name, "work_area": d.work_area,
     "source_type": d.source_type.value, "categories": len(d.categories), "cost_impact_percent": d.cost_impact_percent}
    for d in catalog.drivers
])


,driver_id,name,work_area,source_type,categories,cost_impact_percent
0,agent.verification_loop_complexity,Сложность многоступенчатой верификации агента,experimentation,categorical,4,35.0
1,business.scope_maturity,Зрелость постановки задачи,business_scope,categorical,4,25.0
2,causal_inference.assumption_validation_complexity,Сложность валидации ключевых каузальных предпо...,ml_problem,categorical,4,30.0
3,causal_inference.study_complexity,Сложность настройки и проведения каузального и...,ml_problem,categorical,4,40.0
4,data.adversarial_robustness_requirement,Требования к устойчивости к adversarial атакам,data_quality,categorical,4,25.0
...,...,...,...,...,...,...
74,rollout.deployment_scope,Масштаб rollout,rollout_change_management,categorical,5,20.0
75,scalability.load_level,Уровень нагрузки,scalability,categorical,5,25.0
76,security.access_control,Сложность контроля доступа,security_privacy,categorical,5,18.0
77,serving.delivery_mode,Режим предоставления результата,serving,categorical,5,20.0
